# APs plots in the time domain

In [5]:
BUFFER_BEFORE_SEC = 2
MAX_HOUR_IN_SECONDS = 60*60*4

Required files:
- TTL times
- TTL labels
- APs data (can be read using TTL times)
    - Spike times
    - Spike Clusters
    - Cluster info
    - Templates   
- Sleep/Wake labels
- Video location label

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd

import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

main_folder = r'\\132.66.34.210\Pixel1\1_auditory_neuropixels_BarakH\20240911_C11_T1_NP2_-10dB_g0'
SleepWakeLabels_fname = 'labels_All.mat'
TTL_times_fname = 'TTL_times_adj.txt'
TTL_labels_fname = 'TTL_labels.txt'
AP_spike_times_fname = r'spike_times_sec.npy'
AP_spike_clusters_fname = r'spike_clusters.npy'
AP_cluster_info_fname = r'cluster_info'
AP_templates_fname = r'templates.npy'
VID_locations_fname = r'video_loc_sync.npz'

ClusterInfo_path = find_files(main_folder, AP_cluster_info_fname)[0]
Templates_path = find_files(main_folder, AP_templates_fname)[0]
SpikeTimes_path = find_files(main_folder, AP_spike_times_fname)[0]
SpikeClusters_path = find_files(main_folder, AP_spike_clusters_fname)[0]
TTL_labels_path = find_files(main_folder, TTL_labels_fname)[0]
TTL_times_path = find_files(main_folder, TTL_times_fname)[0]
SleepWakeLabels_path = find_files(main_folder, SleepWakeLabels_fname)[0]
VID_locations_path = find_files(main_folder, VID_locations_fname)[0]

In [7]:
Templates = np.load(Templates_path)
SpikeTimes = np.squeeze(np.load(SpikeTimes_path))
SpikeClusters = np.load(SpikeClusters_path)
ClusterInfo = pd.read_csv(ClusterInfo_path, sep='\t')

TTL_times = np.loadtxt(TTL_times_path)
TTL_labels = np.loadtxt(TTL_labels_path)

SleepWakeLabels =  scipy.io.loadmat(SleepWakeLabels_path)
SleepWakeLabels = np.squeeze(SleepWakeLabels['labels'])
SleepWakeTimes = np.linspace(0, MAX_HOUR_IN_SECONDS, len(SleepWakeLabels))  # the stop value is set by the time I used for manual scoring

VID_location_data = np.load(VID_locations_path)
VID_locations_time = VID_location_data['t_vec']
VID_locations_com = VID_location_data['location_com']
VID_locations_com = np.sqrt(np.abs(VID_locations_com[1:, 0] - VID_locations_com[:-1, 0]) ** 2 + np.abs(VID_locations_com[1:, 1] - VID_locations_com[:-1, 1]) ** 2)
VID_locations_com = VID_locations_com[0:len(VID_locations_time)]

%matplotlib inline
# %matplotlib qt
plt.figure()
plt.plot(VID_locations_time, VID_locations_com, color='blue')
plt.axhline(np.mean(VID_locations_com) + 2 * np.std(VID_locations_com), color='red', linestyle='dashed')
# plt.imshow(VID_location_data['first_frame'])

RATIO_cm_per_pixel = 0.104

## Extract metrics in different time domains

In [8]:
from scipy.stats import wilcoxon, permutation_test
from scipy.ndimage import gaussian_filter
import pandas as pd
import tqdm

# %matplotlib inline
%matplotlib qt

def filter_by_soundType_single(SoundType, TTL_times, SleepWakeTimes, SleepWakeLabels, SpikeTimes, SpikeClusters):
    if SoundType == 100:
        TTL_picked = (TTL_labels > 100) & (TTL_labels < 200)
    elif SoundType == 400:
        TTL_picked = (TTL_labels > 400) & (TTL_labels < 600)
    elif SoundType == 1300:
        TTL_picked = (TTL_labels > 1300) & (TTL_labels < 1400)
    elif SoundType == 1400:
        TTL_picked = (TTL_labels > 1400) & (TTL_labels < 1500)
    else:
        TTL_picked = (TTL_labels == SoundType)
        
    print(f'SoundType {SoundType} - Number of trials: {np.sum(TTL_picked)}')
    cur_SoundType_trialsTimes = TTL_times[TTL_picked]
    
    cur_SoundType_trialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes > 1]
    
    closest_SleepWakeTimes = []
    for curTTLtime in cur_SoundType_trialsTimes:
        closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))
    
    cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]

    closest_MovementTimes = []
    for curTTLtime in cur_SoundType_trialsTimes:
        closest_MovementTimes.append(np.argmin(np.abs(VID_locations_time - curTTLtime)))
        
    VID_locations_com_at_times = VID_locations_com[closest_MovementTimes]
    th_movement = np.mean(VID_locations_com) + 2 * np.std(VID_locations_com)
    cur_SoundType_MovementLabels = VID_locations_com_at_times.copy()
    cur_SoundType_MovementLabels[VID_locations_com_at_times <= th_movement] = 1
    cur_SoundType_MovementLabels[VID_locations_com_at_times > th_movement] = 0
    
    All_TrialsTimes = cur_SoundType_trialsTimes
    First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes < 30*60]
    Last30min_TrialsTimes = cur_SoundType_trialsTimes[(MAX_HOUR_IN_SECONDS - 30 *60) < cur_SoundType_trialsTimes]
    Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
    # Wake_TrialsTimes = cur_SoundType_trialsTimes[(cur_SoundType_SleepWakeLabels == 2)]  ## All Wake
    Wake_TrialsTimes = cur_SoundType_trialsTimes[(cur_SoundType_SleepWakeLabels == 2) & (cur_SoundType_MovementLabels == 1)]  ## Quiet Wake
    # Wake_TrialsTimes = cur_SoundType_trialsTimes[(cur_SoundType_SleepWakeLabels == 2) & (cur_SoundType_MovementLabels == 0)]  ## Active Wake

    Picked_TrialsTimes = All_TrialsTimes
    
    tstep = 3.5 + BUFFER_BEFORE_SEC # Take 3.5 seconds after stimuli
    Trials_read_times_start = Picked_TrialsTimes - BUFFER_BEFORE_SEC # Take N second before Stimuli Onset
    Trials_read_times_end = Trials_read_times_start + tstep # End of trial
    
    Trials_label = list(range(0,len(Picked_TrialsTimes)))
    Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
    Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
    
    mask = np.zeros_like(SpikeTimes, dtype=bool)
    SpikeTrials = np.ones_like(SpikeTimes) * np.nan
    SpikeTimes_inTrials = np.ones_like(SpikeTimes) * np.nan
    for ind, (start, end) in enumerate(zip(Trials_read_times_start, Trials_read_times_end)):
        cur_mask = (SpikeTimes >= start) & (SpikeTimes <= end)
        SpikeTimes_inTrials[cur_mask] = SpikeTimes[cur_mask] - start - BUFFER_BEFORE_SEC
        SpikeTrials[cur_mask] = ind
        mask |= cur_mask
    
    mask = np.squeeze(mask)
    SpikeTimes_inTrials = SpikeTimes_inTrials[mask]
    SpikeClusters_inTrials = SpikeClusters[mask]
    SpikeTrials_inTrials = SpikeTrials[mask]
    
    return SpikeTimes_inTrials, SpikeClusters_inTrials, SpikeTrials_inTrials, Trials_sleep, Trials_Wake, Trials_label, VID_locations_com_at_times

def filter_by_cluster_single(cluster_pick, SpikeTimes_inTrials, SpikeTrials_inTrials, SpikeClusters_inTrials, Trials_label, t_vec):
    Spikes_array = np.zeros((len(Trials_label), len(t_vec)))

    SpikeTimes_inTrials_plot = SpikeTimes_inTrials[SpikeClusters_inTrials == cluster_pick]
    SpikeTrials_inTrials_plot = SpikeTrials_inTrials[SpikeClusters_inTrials == cluster_pick].astype(int)
    
    time_bin_indices = np.searchsorted(t_vec, SpikeTimes_inTrials_plot) - 1
    time_bin_indices = np.clip(time_bin_indices, 0, len(t_vec) - 1)
    Spikes_array[SpikeTrials_inTrials_plot, time_bin_indices] = 1
    
    # Accumulate the spike counts
    # np.add.at(Spikes_array, (SpikeTrials_inTrials_plot, time_bin_indices), 1)
    
    return Spikes_array

def statistic_test(x, y):
    # return wilcoxon(x, y, zero_method='pratt', method='auto').statistic
    return np.abs(x.mean() - y.mean())
    # return np.abs(np.mean(x - y))

def extract_params(Spikes_Array, t_vec, SoundType, Trials):
    Spontaneous_times = [-0.5, 0]
    Onset_times = [0, 0.100]
    PostOnset_times = [0.100, 0.200]  # [0.03, 0.08]
    Sustained_times = [0.200, 2.500]  ## Only for [101, 400]
    Offset_times = [2.550, 2.650]  ## Only for [101, 400, 1300]
    PVALUE_TH = 0.1
    
    PSTH = Spikes_Array[Trials, :] / (t_vec[1] - t_vec[0])  # FR (Spikes/Sec)
    sigma_gauss = 0.002 / (t_vec[1] - t_vec[0])
    PSTH_filtered = gaussian_filter(PSTH, sigma=sigma_gauss, axes=[1])
    
    Spontaneous_array = PSTH_filtered[:, (t_vec>=Spontaneous_times[0]) & (t_vec<Spontaneous_times[1])]
    Spontaneous_FR_per_trial = np.mean(Spontaneous_array, axis=1) 
    Spontaneous_FR = np.mean(Spontaneous_FR_per_trial, axis=0) 
    Spontaneous_STD = (np.std(np.mean(Spontaneous_array, axis=0), axis=0) + 0.001)
    
    Onset_array = PSTH_filtered[:, (t_vec>=Onset_times[0]) & (t_vec<Onset_times[1])]
    t_vec_onset = t_vec[(t_vec>=Onset_times[0]) & (t_vec<Onset_times[1])]
    
    Onset_FR_per_trial = np.mean(Onset_array, axis=1)
    Onset_array_mean = np.mean(Onset_array, axis=0)
    max_FR = np.max(Onset_array_mean, axis=0)
    min_FR = np.min(Onset_array_mean, axis=0)
    Onset_FR = max_FR if (max_FR - Spontaneous_FR) > (min_FR - Spontaneous_FR) else min_FR 
    Onset_FR_mean = np.mean(Onset_FR_per_trial)
    Onset_Latency = t_vec_onset[np.argmax(Onset_array_mean, axis=0) if (max_FR - Spontaneous_FR) > (min_FR - Spontaneous_FR) else np.argmin(Onset_array_mean, axis=0)]
    
    # Onset_max_FR_per_trial = np.max(Onset_array, axis=1)
    # Onset_min_FR_per_trial = np.min(Onset_array, axis=1)
    # t_max_per_trial = t_vec_onset[np.argmax(Onset_array, axis=1)]
    # t_min_per_trial = t_vec_onset[np.argmin(Onset_array, axis=1)]
    # diff_max = np.abs(Onset_max_FR_per_trial - Spontaneous_FR_per_trial)
    # diff_min = np.abs(Onset_min_FR_per_trial - Spontaneous_FR_per_trial)
    # 
    # Onset_FR_per_trial = np.where(diff_max > diff_min, Onset_max_FR_per_trial, Onset_min_FR_per_trial)
    # t_onset_per_trial = np.where(diff_max > diff_min, t_max_per_trial, t_min_per_trial)
    # 
    # Onset_FR = np.mean(Onset_FR_per_trial, axis=0)
    # Onset_Latency = np.mean(t_onset_per_trial, axis=0)
    
    '''Remove Baseline'''
    # Onset_FR_per_trial = Onset_FR_per_trial - Spontaneous_FR_per_trial
    # Onset_FR = Onset_FR - Spontaneous_FR

    if not np.sum(Spontaneous_FR_per_trial-Onset_FR_per_trial) == 0:
        Onset_stat = wilcoxon(Spontaneous_FR_per_trial, Onset_FR_per_trial, zero_method='pratt', method='auto')
        # Onset_stat = permutation_test((Spontaneous_FR_per_trial, Onset_FR_per_trial), statistic_test, permutation_type='samples', alternative='greater', n_resamples=9999)
        # hist, bin_edges = np.histogram(Onset_stat.null_distribution, bins=50)
        # plt.figure()
        # plt.hist(Onset_stat.null_distribution, bins=50)
        # plt.axvline(x=Onset_stat.statistic)
        # plt.title("Permutation distribution of test statistic")
        # plt.xlabel("Value of Statistic")
        # plt.ylabel("Frequency")
        # plt.show(block=False)
        
        # Onset_pvalue = np.sum(Onset_stat.null_distribution > Onset_stat.statistic) / len(Onset_stat.null_distribution)
        Onset_pvalue = Onset_stat.pvalue
    else:
       Onset_pvalue = None 
    
    # if Onset_pvalue < PVALUE_TH:
    PostOnset_array = PSTH_filtered[:, (t_vec>=PostOnset_times[0]) & (t_vec<PostOnset_times[1])]
    PostOnset_FR_per_trial = np.mean(PostOnset_array[:, :], axis=1) 
    PostOnset_FR = np.mean(PostOnset_FR_per_trial, axis=0) 
    
    '''Remove Baseline'''
    # PostOnset_FR_per_trial = PostOnset_FR_per_trial - Spontaneous_FR_per_trial
    # PostOnset_FR = PostOnset_FR - Spontaneous_FR
    
    if not np.sum(Spontaneous_FR_per_trial-PostOnset_FR_per_trial) == 0:
        PostOnset_stat = wilcoxon(Spontaneous_FR_per_trial, PostOnset_FR_per_trial, zero_method='pratt', method='auto')
        # PostOnset_stat = permutation_test((Spontaneous_FR_per_trial, PostOnset_FR_per_trial), statistic_test, permutation_type='samples', alternative='two-sided', n_resamples=5000)
        PostOnset_pvalue = PostOnset_stat.pvalue
    else:
        PostOnset_pvalue = None
  
    Sustained_array = PSTH_filtered[:, (t_vec>=Sustained_times[0]) & (t_vec<Sustained_times[1])]
    Sustained_FR_per_trial = np.mean(Sustained_array, axis=1)
    Sustained_FR = np.mean(Sustained_FR_per_trial, axis=0) 
    
    '''Remove Baseline'''
    # Sustained_FR_per_trial = Sustained_FR_per_trial - Spontaneous_FR_per_trial
    # Sustained_FR = Sustained_FR - Spontaneous_FR
    
    if not np.sum(Spontaneous_FR_per_trial-Sustained_FR_per_trial) == 0:
        Sustained_stat = wilcoxon(Spontaneous_FR_per_trial, Sustained_FR_per_trial, zero_method='pratt', method='auto')
        # Sustained_stat = permutation_test((Spontaneous_FR_per_trial, Sustained_FR_per_trial), statistic_test, permutation_type='samples', alternative='two-sided', n_resamples=5000)
        Sustained_pvalue = Sustained_stat.pvalue
    else:
        Sustained_pvalue = None

     # if SoundType == 101 or SoundType == 400:
        #     if SoundType == 101:
        #         inter_click = 1 / 40  ## sec between intervals
        #     elif SoundType == 400:
        #         inter_click = 1 / 25  ## sec between intervals    
        #     Sustained_array = PSTH_filtered[:, (t_vec>=Sustained_times[0]) & (t_vec<Sustained_times[1])]
        #     inter_click_ind = int(inter_click / (t_vec[1] - t_vec[0]))
        #     Sustained_array_shape1_crop = Sustained_array.shape[1] // inter_click_ind
        #     Sustained_array_reshaped = np.reshape(Sustained_array[:,:int(Sustained_array_shape1_crop * inter_click_ind)], (Sustained_array.shape[0], inter_click_ind, -1))
        #     Sustained_FR_per_phase = np.mean(Sustained_array_reshaped, axis=2)
        #     Sustained_FR_per_phase_mean_trials = np.mean(Sustained_FR_per_phase, axis=0)
        #     Sustained_FR = np.max(Sustained_FR_per_phase_mean_trials, axis=0) - np.min(Sustained_FR_per_phase_mean_trials, axis=0)
        #     Sustained_FR_per_trial = np.mean(Sustained_array, axis=1)
        
    if SoundType == 101 or SoundType == 400 or SoundType == 1300:
        Offset_array = PSTH_filtered[:, (t_vec>=Offset_times[0]) & (t_vec<Offset_times[1])]
        Offset_FR_per_trial = np.mean(Offset_array[:, :], axis=1) 
        Offset_FR = np.mean(Offset_FR_per_trial, axis=0)
        
        '''Remove Baseline'''
        # Offset_FR_per_trial = Offset_FR_per_trial - Spontaneous_FR_per_trial
        # Offset_FR = Offset_FR - Spontaneous_FR
        
        if not np.sum(Spontaneous_FR_per_trial-Offset_FR_per_trial) == 0:
            Offset_stat = wilcoxon(Spontaneous_FR_per_trial, Offset_FR_per_trial, zero_method='pratt', method='auto')
            # Offset_stat = permutation_test((Spontaneous_FR_per_trial, Offset_FR_per_trial), statistic_test, permutation_type='samples', alternative='two-sided', n_resamples=5000)
            Offset_pvalue = Offset_stat.pvalue
        else:
            Offset_pvalue = None

    else:
        Offset_FR = None
        Offset_pvalue = None
    return PSTH_filtered, Spontaneous_FR, Onset_FR, Onset_Latency, Onset_pvalue, PostOnset_FR, PostOnset_pvalue, Sustained_FR, Sustained_pvalue, Offset_FR, Offset_pvalue, Spontaneous_STD, Onset_FR_mean, Spikes_Array[Trials, :]
    # else:
    #     return 'skip'
        
        
df = pd.DataFrame({"SoundType":[], "Channel": [], "State": "", "Cluster": [], "Spontaneous_FR": [], "Spontaneous_STD": [], "Onset_FR": [], "Onset_Latency": [], "Onset_pvalue": [], "PostOnset_FR": [], "PostOnset_pvalue": [], "Sustained_FR": [], "Sustained_pvalue": [], "Offset_FR": [], "Offset_pvalue": [], "Movement": [], "PSTH": []})

## TODO: "ON-State/OFF-State"

# SoundTypes = [101, 102]
SoundTypes = [101, 102, 400, 1300]
# cluster_picks = [96]

good_neurons = (ClusterInfo['KSLabel'] == 'good')
# good_neurons = (ClusterInfo['group'] == 'good')

cluster_picks = ClusterInfo.loc[good_neurons,'cluster_id'].unique()
t_vec = np.arange(-BUFFER_BEFORE_SEC, 3.5, 0.001)

for SoundType in SoundTypes:
    SpikeTimes_inTrials, SpikeClusters_inTrials, SpikeTrials_inTrials, Trials_sleep, Trials_Wake, Trials_label, VID_com_trials = filter_by_soundType_single(SoundType, TTL_times, SleepWakeTimes, SleepWakeLabels, SpikeTimes, SpikeClusters)
    for cluster_pick in tqdm.tqdm(cluster_picks):
        ClusterInfo_clusterpick = ClusterInfo[ClusterInfo['cluster_id'] == cluster_pick]
        # if ClusterInfo_clusterpick['group'].values[0] == 'good':
        if ClusterInfo_clusterpick['KSLabel'].values[0] == 'good':
            Spikes_array = filter_by_cluster_single(cluster_pick, SpikeTimes_inTrials, SpikeTrials_inTrials, SpikeClusters_inTrials, Trials_label, t_vec)
            # PSTH, Spontaneous_FR, Onset_FR, Onset_pvalue, PostOnset_FR, PostOnset_pvalue, Sustained_FR, Sustained_pvalue, Offset_FR, Offset_pvalue = extract_params(Spikes_array, t_vec, SoundType, Trials_label)
            # new_row_df = pd.DataFrame([{"SoundType":int(SoundType), "Channel": int(ClusterInfo_clusterpick['ch'].iloc[0]), "State": "All", "Cluster": cluster_pick, "PSTH": PSTH, "Spontaneous_FR": Spontaneous_FR, "Onset_FR": Onset_FR, "Onset_pvalue": Onset_pvalue, "PostOnset_FR": PostOnset_FR, "PostOnset_pvalue": PostOnset_pvalue, "Sustained_FR": Sustained_FR, "Sustained_pvalue": Sustained_pvalue, "Offset_FR": Offset_FR, "Offset_pvalue": Offset_pvalue}])
            # df = pd.concat([df.loc[:, df.notna().any(axis=0)], new_row_df.loc[:, new_row_df.notna().any(axis=0)]])   
            
            result_sleep = extract_params(Spikes_array, t_vec, SoundType, np.squeeze(Trials_sleep))
            result_wake = extract_params(Spikes_array, t_vec, SoundType, np.squeeze(Trials_Wake))
            # PSTH_filtered, Spontaneous_FR, Onset_FR, Onset_pvalue, PostOnset_FR, PostOnset_pvalue, Sustained_FR, Sustained_pvalue, Offset_FR, Offset_pvalue
            
            if not (result_sleep == "skip" or result_wake == "skip"):
                new_row_df_sleep = pd.DataFrame([{"SoundType":int(SoundType), "Channel": int(ClusterInfo_clusterpick['ch'].iloc[0]), "State": "Sleep", "Cluster": cluster_pick, "Spontaneous_FR": result_sleep[1], "Spontaneous_STD": result_sleep[11], "Onset_FR": result_sleep[2], "Onset_FR_mean": result_sleep[12],"Onset_Latency": result_sleep[3], "Onset_pvalue": result_sleep[4], "PostOnset_FR": result_sleep[5], "PostOnset_pvalue": result_sleep[6], "Sustained_FR": result_sleep[7], "Sustained_pvalue": result_sleep[8], "Offset_FR": result_sleep[9], "Offset_pvalue": result_sleep[10], "Movement": VID_com_trials[np.squeeze(Trials_sleep)], "PSTH": result_sleep[0], "Spikes_raw": result_sleep[13]}])
                df = pd.concat([df.loc[:, df.notna().any(axis=0)], new_row_df_sleep.loc[:, new_row_df_sleep.notna().any(axis=0)]])   
    
                new_row_df_wake = pd.DataFrame([{"SoundType":int(SoundType), "Channel": int(ClusterInfo_clusterpick['ch'].iloc[0]), "State": "Wake", "Cluster": cluster_pick, "Spontaneous_FR": result_wake[1], "Spontaneous_STD": result_wake[11], "Onset_FR": result_wake[2], "Onset_FR_mean": result_wake[12], "Onset_Latency": result_wake[3], "Onset_pvalue": result_wake[4], "PostOnset_FR": result_wake[5], "PostOnset_pvalue": result_wake[6], "Sustained_FR": result_wake[7], "Sustained_pvalue": result_wake[8], "Offset_FR": result_wake[9], "Offset_pvalue": result_wake[10], "Movement": VID_com_trials[np.squeeze(Trials_Wake)], "PSTH": result_wake[0], "Spikes_raw": result_wake[13]}])
                df = pd.concat([df.loc[:, df.notna().any(axis=0)], new_row_df_wake.loc[:, new_row_df_wake.notna().any(axis=0)]])   
            

In [9]:
display(df)

### Amit Plot - Compare metrics according to selected significant metric

In [12]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
# %matplotlib qt

def filter_dataframe(df, sound_pick, state, channels):
    return df[(df['SoundType'].isin(sound_pick)) & (df['State'] == state) & 
              (df['Channel'].between(channels[0], channels[-1]))
                ].copy()

def get_significant_df(df_Sleep, df_Wake, pval_check, stateOI):
    condition_STD = (np.abs(df_Sleep[f'{stateOI}_FR'] - df_Sleep['Spontaneous_FR']) > 1 * df_Sleep['Spontaneous_STD']) | (np.abs(df_Wake[f'{stateOI}_FR'] - df_Wake['Spontaneous_FR']) > 1 * df_Wake['Spontaneous_STD'])
    condition_pval = (df_Sleep[f'{stateOI}_pvalue'].between(pval_check[0], pval_check[1])) | (df_Wake[f'{stateOI}_pvalue'].between(pval_check[0], pval_check[1]))
    condition_valence = ((df_Sleep[f'{stateOI}_FR'] - df_Sleep['Spontaneous_FR']) > 0) | ((df_Wake[f'{stateOI}_FR'] - df_Wake['Spontaneous_FR']) > 0)
    
    condition = condition_pval & condition_valence & condition_STD
    # condition = condition_pval & condition_valence
    return df_Sleep[condition].reset_index(drop=True), df_Wake[condition].reset_index(drop=True)

sound_pick = [102]
pval_check = [-1, 0.01]
channels_pick = [1, 384]
plot_type = 'ModIndex'  # 'ModIndex', 'FR'
time_periods = {
    "Spontaneous_times": [-0.5, 0],
    "Onset_times": [0, 0.100],
    "PostOnset_times": [0.100, 0.200],
    "Sustained_times": [0.200, 2.5],  # Only for [101, 400]
    "Offset_times": [2.55, 2.65]  # Only for [101, 400, 1300]
}
stateOI = 'Onset'

# Filter dataframes for each state
df_Sleep = filter_dataframe(df, sound_pick, 'Sleep', channels_pick)
df_Wake = filter_dataframe(df, sound_pick, 'Wake', channels_pick)
display(df_Wake)
df_Sleep, df_Wake = get_significant_df(df_Sleep, df_Wake, pval_check, stateOI)

# Define plotting function
def plot_bar(ax, df_sleep, df_wake, fr_column, color, label, channels_to_look, plot_type, cluster_list):
    if plot_type == 'ModIndex':
        mean_diff = ((df_sleep[fr_column] - df_wake[fr_column]) / np.maximum(np.abs(df_sleep[fr_column]), np.abs(df_wake[fr_column])) * 100).mean()
        bars = ax.bar(df_sleep['Channel'], 
                      (df_sleep[fr_column] - df_wake[fr_column]) / (np.maximum(np.abs(df_sleep[fr_column]), np.abs(df_wake[fr_column]))) * 100, 
                      color=color, label=label, alpha=0.5, width=2)
        ax.set_ylabel('ModIndex')
        ax.set_ylim([-100, 100])
    else:
        mean_diff = (df_sleep[fr_column] - df_wake[fr_column]).mean()
        bars = ax.bar(df_sleep['Channel'], 
                      (df_sleep[fr_column] - df_wake[fr_column]), 
                      color=color, label=label, alpha=0.5, width=2)
        ax.set_ylabel('FR')
        ax.set_ylim([-15, 15])
    
    rects = ax.patches
    
    # Highlight the bars corresponding to the clusters picked in the raster plot
    for rect, bar, cluster in zip(rects, bars, df_sleep['Cluster']):
        if cluster in cluster_list:
            bar.set_edgecolor('black')
            bar.set_linewidth(2)
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2, height + np.sign(height) * 5, cluster, ha="center", va="bottom")

    # ax.legend(loc='upper right')
    ax.set_xlabel('Channel')
    ax.set_xlim(channels_to_look)
    
    # Add mean plot
    ax.axhline(y=mean_diff, color='black', linestyle='--', linewidth=2, label=f'Mean {label}')
    # ax.legend(loc='upper right')
    ax.set_title(f'{len(df_sleep)} - {fr_column}')

def plot_raster(ax, df_sleep, df_wake, t_vec, cluster_pick, time_periods):
    clus_line = df_sleep['Cluster'] == cluster_pick
    if clus_line.sum() == 0:
        raise ValueError(f"No cluster found with the index {cluster_pick}")
    
    cur_channel = df_sleep.loc[clus_line, 'Channel'].values[0]
    cur_sound = df_sleep.loc[clus_line, 'SoundType'].values[0]
    cur_sleep = df_sleep.loc[clus_line, 'PSTH'].values[0]
    cur_wake = df_wake.loc[clus_line, 'PSTH'].values[0]
    cur_raster = np.invert(np.vstack((cur_sleep, cur_wake)) > 0)
    y_vec = np.arange(cur_raster.shape[0])
    extent_vec = (min(t_vec), max(t_vec), 0, cur_raster.shape[0])
    ax.imshow(cur_raster, extent=extent_vec, aspect='auto', cmap='gray', vmin=0, vmax=1, origin='lower')
    ax.set_yticks([cur_sleep.shape[0]/2, cur_sleep.shape[0] + cur_wake.shape[0]/2], ['Sleep', 'Wake'])
    ax.set_xlim([-0.5, 1.0])
    # ax.set_xlim([-0.5, 0.5])
    ax.set_ylim([np.min(y_vec), np.max(y_vec)])
    ax.set_xlabel('Time [s]')

    ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax.axhline(y=cur_sleep.shape[0], color='blue', linestyle='-', linewidth=4)

    # Fill between for each time period
    colors = {
        "Spontaneous_times": 'green',
        "Onset_times": 'blue',
        "PostOnset_times": 'yellow',
        "Sustained_times": 'red',
        "Offset_times": 'gray'
    }
    for period, times in time_periods.items():
        color = colors.get(period)  # Default color is yellow if period is not in colors
        ax.fill_betweenx(y_vec, times[0], times[1], color=color, alpha=0.2)
    
    # Extract movement data for sleep and wake
    movement_sleep = df_sleep.loc[clus_line, 'Movement'].values[0]
    movement_wake = df_wake.loc[clus_line, 'Movement'].values[0]
    
    # Create a combined movement array and corresponding trial indices
    movement_data = np.hstack((movement_sleep, movement_wake)) * RATIO_cm_per_pixel
    movement_trials = np.arange(len(movement_data))
    
    # Create a secondary x-axis for movement data
    ax2 = ax.twiny()
    ax2.plot(movement_data, movement_trials, color='red', alpha=0.4, linestyle='--')
    
    # Set the limits and labels for the secondary x-axis
    ax2.set_xlim([0, 2])
    ax2.set_xlabel('Movement Data [cm]')

    ax.set_title(f'Channel {cur_channel}, Cluster {cluster_pick}, Sound {cur_sound}')

def plot_PSTH(ax, df_sleep, df_wake, t_vec, cluster_pick, time_periods):
    clus_line = df_sleep['Cluster'] == cluster_pick
    if clus_line.sum() == 0:
        raise ValueError(f"No cluster found with the index {cluster_pick}")
    
    cur_sleep = gaussian_filter(df_sleep.loc[clus_line, 'PSTH'].values[0].mean(axis=0), 8)
    cur_wake = gaussian_filter(df_wake.loc[clus_line, 'PSTH'].values[0].mean(axis=0), 8)
    # cur_sleep = cur_sleep - cur_sleep[t_vec<0].mean()
    # cur_wake = cur_wake - cur_wake[t_vec<0].mean()
    
    ax.plot(t_vec, cur_sleep, color='red', label='Sleep')
    ax.plot(t_vec, gaussian_filter(df_sleep.loc[clus_line, 'PSTH'].values[0].T, 8), color='red', label='Sleep-All', alpha=0.01)
    ax.plot(t_vec, cur_wake, color='green', label='Wake')
    ax.plot(t_vec, gaussian_filter(df_wake.loc[clus_line, 'PSTH'].values[0].T, 8), color='green', label='Wake-All', alpha=0.01)

    y_vec = np.arange(np.min((cur_sleep, cur_wake)) - 2, np.max((cur_sleep, cur_wake)) + 2)
    ax.set_xlim([-0.5, 1.0])
    # ax.set_xlim([-0.5, 0.5])    
    # ax.set_title(f'Sleep {df_sleep.loc[clus_line, "Onset_pvalue"].values[0]}, Wake {df_wake.loc[clus_line, "Onset_pvalue"].values[0]}')

    # Fill between for each time period
    colors = {
        "Spontaneous_times": 'green',
        "Onset_times": 'blue',
        "PostOnset_times": 'yellow',
        "Sustained_times": 'red',
        "Offset_times": 'gray'
    }
    for period, times in time_periods.items():
        color = colors.get(period)
        ax.fill_betweenx(y_vec, times[0], times[1], color=color, alpha=0.2)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax.axvline(x=df_sleep.loc[clus_line, "Onset_Latency"].values[0], color='red', linestyle='-.', linewidth=1)
    ax.axvline(x=df_wake.loc[clus_line, "Onset_Latency"].values[0], color='green', linestyle='-.', linewidth=1)

    ax.set_ylim([y_vec.min(), y_vec.max()])

display(df_Wake.sort_values('Channel', ascending=False, inplace=False).reset_index(drop=True))
display(df_Sleep.sort_values('Channel', ascending=False, inplace=False).reset_index(drop=True))

cluster_list_PSTH = df_Wake.sort_values('Onset_FR', ascending=False, inplace=False).loc[:, 'Cluster'].unique()
cluster_list_PSTH = cluster_list_PSTH[[0,1,2,3,4]]
display(cluster_list_PSTH)

# Plotting
fig = plt.figure(figsize=(16, 12), tight_layout=False)

ax1 = plt.subplot2grid((4, 5), (0, 0), colspan=1)
plot_bar(ax1, df_Sleep, df_Wake, 'Spontaneous_FR', 'green', 'Spontaneous FR', channels_pick, plot_type, cluster_list_PSTH)

ax2 = plt.subplot2grid((4, 5), (0, 1), colspan=1)
plot_bar(ax2, df_Sleep, df_Wake, 'Onset_FR', 'blue', 'Onset FR', channels_pick, plot_type, cluster_list_PSTH)

ax3 = plt.subplot2grid((4, 5), (0, 2), colspan=1)
plot_bar(ax3, df_Sleep, df_Wake, 'PostOnset_FR', 'yellow', 'PostOnset FR', channels_pick, plot_type, cluster_list_PSTH)

ax4 = plt.subplot2grid((4, 5), (0, 3), colspan=1)
plot_bar(ax4, df_Sleep, df_Wake, 'Sustained_FR', 'red', 'Sustained FR', channels_pick, plot_type, cluster_list_PSTH)

ax5 = plt.subplot2grid((4, 5), (0, 4), colspan=1)
plot_bar(ax5, df_Sleep, df_Wake, 'Offset_FR', 'gray', 'Offset FR', channels_pick, plot_type, cluster_list_PSTH)

for ii, cluster in enumerate(cluster_list_PSTH):
    ax_raster = plt.subplot2grid((4, 5), (1, ii), colspan=1, rowspan=2)
    plot_raster(ax_raster, df_Sleep, df_Wake, t_vec, cluster, time_periods)
    ax_psth = plt.subplot2grid((4, 5), (3, ii), colspan=1, rowspan=1)
    plot_PSTH(ax_psth, df_Sleep, df_Wake, t_vec, cluster, time_periods)

plt.suptitle(f'{sound_pick}, chs {channels_pick}, p_val th {pval_check}')

# fig = plt.figure(figsize=(5, 5), tight_layout=True)
# ax1 = plt.subplot2grid((1, 1), (0, 0), colspan=1)
# ax1.scatter(df_Wake['Onset_Latency'] * 1000, df_Wake['Onset_FR'], color='green', label='Wake')
# ax1.scatter(df_Sleep['Onset_Latency'] * 1000, df_Sleep['Onset_FR'], color='red', label='Sleep')
# ax1.axvline(x=df_Sleep['Onset_Latency'].mean() * 1000, color='red', linestyle='--', linewidth=1)
# ax1.axvline(x=df_Wake['Onset_Latency'].mean() * 1000, color='green', linestyle='--', linewidth=1)
# ax1.set_xlabel('Onset Latency (ms)')
# ax1.set_ylabel('Onset FR (Spikes/s)')
# ax1.set_xlim([0, 100])
# ax1.legend()

if plot_type == 'ModIndex':
    Spont_bar = ((df_Sleep['Spontaneous_FR'] - df_Wake['Spontaneous_FR']) / np.maximum(np.abs(df_Sleep['Spontaneous_FR']), np.abs(df_Wake['Spontaneous_FR'])) * 100)
    Onset_bar = ((df_Sleep['Onset_FR'] - df_Wake['Onset_FR']) / np.maximum(np.abs(df_Sleep['Onset_FR']), np.abs(df_Wake['Onset_FR'])) * 100)
    Postonset_bar = ((df_Sleep['PostOnset_FR'] - df_Wake['PostOnset_FR']) / np.maximum(np.abs(df_Sleep['PostOnset_FR']), np.abs(df_Wake['PostOnset_FR'])) * 100)
    Sustained_bar = ((df_Sleep['Sustained_FR'] - df_Wake['Sustained_FR']) / np.maximum(np.abs(df_Sleep['Sustained_FR']), np.abs(df_Wake['Sustained_FR'])) * 100)
    Offset_bar = ((df_Sleep['Offset_FR'] - df_Wake['Offset_FR']) / np.maximum(np.abs(df_Sleep['Offset_FR']), np.abs(df_Wake['Offset_FR'])) * 100)
else:
    Spont_bar = (df_Sleep['Spontaneous_FR'] - df_Wake['Spontaneous_FR'])
    Onset_bar = (df_Sleep['Onset_FR'] - df_Wake['Onset_FR'])
    Postonset_bar = (df_Sleep['PostOnset_FR'] - df_Wake['PostOnset_FR'])
    Sustained_bar = (df_Sleep['Sustained_FR'] - df_Wake['Sustained_FR'])
    Offset_bar = (df_Sleep['Offset_FR'] - df_Wake['Offset_FR'])

# Calculate means
mean_Spont_bar = np.mean(Spont_bar)
mean_Onset_bar = np.mean(Onset_bar)
mean_Postonset_bar = np.mean(Postonset_bar)
mean_Sustained_bar = np.mean(Sustained_bar)
mean_Offset_bar = np.mean(Offset_bar)

# Create figure and axis
fig = plt.figure(figsize=(5, 5), tight_layout=True)
ax1 = plt.subplot2grid((1, 1), (0, 0), colspan=1)

# Create bar plot
bar_positions = np.arange(5)
bar_heights = [mean_Spont_bar, mean_Onset_bar, mean_Postonset_bar, mean_Sustained_bar, mean_Offset_bar]
ax1.bar(bar_positions, bar_heights, color=['gray', 'gray', 'green', 'yellow', 'magenta'], alpha=0.7)

# Add scatter plot
ax1.scatter(np.ones_like(Spont_bar) * 0, Spont_bar, color='blue', s=0.5)
ax1.scatter(np.ones_like(Onset_bar) * 1, Onset_bar, color='blue', s=0.5)
ax1.scatter(np.ones_like(Postonset_bar) * 2, Postonset_bar, color='blue', s=0.5)
ax1.scatter(np.ones_like(Sustained_bar) * 3, Sustained_bar, color='blue', s=0.5)
ax1.scatter(np.ones_like(Sustained_bar) * 4, Offset_bar, color='blue', s=0.5)

# Customize the plot
ax1.set_xticks(bar_positions)
ax1.set_xticklabels(['Spontaneous', 'Onset', 'Postonset', 'Sustained', 'Offset'])
if plot_type == 'ModIndex':
    ax1.set_ylabel('ModIndex')
else:
    ax1.set_ylabel('Diff FR (SLEEP-WAKE)')


### Yaniv Plot - latency vs gain at sleep and wake

In [13]:
from scipy.stats import pearsonr

sound_pick = [102, 101]
pval_check = [-1, 0.01]
stateOI = 'Onset'

# Filter dataframes for each state
df_Sleep = filter_dataframe(df, sound_pick, 'Sleep', [1, 150])
df_Wake = filter_dataframe(df, sound_pick, 'Wake', [1, 150])

df_Sleep, df_Wake = get_significant_df(df_Sleep, df_Wake, pval_check, stateOI)
display(df_Sleep)

channels = df_Sleep['Channel']
latency_sleep = df_Sleep['Onset_Latency'] * 1000
latency_wake = df_Wake['Onset_Latency'] * 1000

FR_onset_sleep = np.abs(df_Sleep[f'{stateOI}_FR'] - df_Sleep['Spontaneous_FR'])
FR_onset_wake = np.abs(df_Wake[f'{stateOI}_FR'] - df_Sleep['Spontaneous_FR'])

Gain_FR = (FR_onset_sleep - FR_onset_wake) / np.maximum(np.abs(FR_onset_sleep), np.abs(FR_onset_wake)) * 100

# Remove inf and NaN values
valid_indices = np.isfinite(latency_sleep) & np.isfinite(Gain_FR) & np.isfinite(latency_wake)
latency_sleep = latency_sleep[valid_indices]
latency_wake = latency_wake[valid_indices]
Gain_FR = Gain_FR[valid_indices]
channels = channels[valid_indices]

# Linear fit for combined data
m_sleep, b_sleep = np.polyfit(latency_sleep, Gain_FR, 1)
m_wake, b_wake = np.polyfit(latency_wake, Gain_FR, 1)

# Correlation for combined data
r_sleep, p_sleep = pearsonr(latency_sleep, Gain_FR)
r_wake, p_wake = pearsonr(latency_wake, Gain_FR)

fig = plt.figure(figsize=(18, 9))
ax1 = plt.subplot2grid((2, 4), (0, 0), colspan=1)

ax1.scatter(latency_sleep[channels.between(120, 140)], Gain_FR[channels.between(120, 140)], color='orange', label='low-sleep')
ax1.scatter(latency_sleep[channels.between(1, 120)], Gain_FR[channels.between(1, 120)], color='blue', label='high-sleep')
ax1.plot(latency_sleep, m_sleep * latency_sleep + b_sleep, color='red', linestyle='--')
t = ax1.text(0.20, 0.05, f'r = {r_sleep:.2f} p = {p_sleep:.4f}', transform=ax1.transAxes, color='red', verticalalignment='top')
t.set_bbox(dict(facecolor='black', alpha=0.5, edgecolor='red'))
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel(f'{stateOI} FR SLEEP-WAKE (Spikes/s)')
ax1.legend(loc='upper right')
ax1.set_ylim([-120, 120])

ax1 = plt.subplot2grid((2, 4), (0, 1), colspan=1)
ax1.scatter(latency_wake[channels.between(120, 140)], Gain_FR[channels.between(120, 140)], color='orange', label='low-wake')
ax1.scatter(latency_wake[channels.between(1, 120)], Gain_FR[channels.between(1, 120)], color='blue', label='high-wake')
ax1.plot(latency_wake, m_wake * latency_wake + b_wake, color='green', linestyle='--')
t = ax1.text(0.20, 0.05, f'r = {r_wake:.2f} p = {p_wake:.4f}', transform=ax1.transAxes, color='green', verticalalignment='top')
t.set_bbox(dict(facecolor='black', alpha=0.5, edgecolor='red'))
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel(f'{stateOI} FR SLEEP-WAKE (Spikes/s)')
ax1.legend(loc='upper right')
ax1.set_ylim([-120, 120])

ax1 = plt.subplot2grid((2, 4), (1, 0), colspan=2)
ax1.scatter(latency_sleep[channels.between(120, 140)] - latency_wake[channels.between(120, 140)], Gain_FR[channels.between(120, 140)], color='orange', label='low')
ax1.scatter(latency_sleep[channels.between(1, 120)] - latency_wake[channels.between(1, 120)], Gain_FR[channels.between(1, 120)], color='blue', label='high')
ax1.set_xlabel('Latency SLEEP-WAKE (ms)')
ax1.set_ylabel(f'{stateOI} FR SLEEP-WAKE (Spikes/s)')
ax1.legend(loc='upper right')

ax2 = plt.subplot2grid((2, 4), (0, 2), colspan=1)
ax2.scatter(latency_sleep, channels, color='green')
ax2.set_ylabel('Channel')
ax2.set_xlabel('Latency (ms)')
ax2 = plt.subplot2grid((2, 4), (0, 3), colspan=1)
ax2.scatter(latency_wake, channels, color='red')
ax2.set_ylabel('Channel')
ax2.set_xlabel('Latency (ms)')

ax2 = plt.subplot2grid((2, 4), (1, 2), colspan=2)
ax2.scatter(latency_sleep - latency_wake, channels)
ax2.set_ylabel('Channel')
ax2.set_xlabel('Latency SLEEP-WAKE (ms)')

## Latency distributions- Sleep vs. Wake

In [14]:
sound_pick = [102, 101]
pval_check = [-1, 0.01]
stateOI = 'Onset'

# Filter dataframes for each state
df_Sleep = filter_dataframe(df, sound_pick, 'Sleep', [1, 120])
df_Wake = filter_dataframe(df, sound_pick, 'Wake', [1, 120])

df_Sleep, df_Wake = get_significant_df(df_Sleep, df_Wake, pval_check, stateOI)

channels = df_Sleep['Channel']
latency_sleep = df_Sleep['Onset_Latency'] * 1000
latency_wake = df_Wake['Onset_Latency'] * 1000
bins_latency = np.linspace(np.min(np.minimum(latency_sleep, latency_wake)), np.max(np.maximum(latency_sleep, latency_wake)), 50)

FR_state_sleep = np.abs(df_Sleep[f'{stateOI}_FR'] - df_Sleep['Spontaneous_FR'])
FR_state_wake = np.abs(df_Wake[f'{stateOI}_FR'] - df_Sleep['Spontaneous_FR'])
bins_FR = np.linspace(np.min(np.minimum(FR_state_sleep, FR_state_wake)), np.max(np.maximum(FR_state_sleep, FR_state_wake)), 50)

plt.figure(figsize=(15, 10))
ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=1)
ax1.hist(FR_state_sleep, bins=bins_FR, color='red', label='Sleep', alpha=1)
ax1.hist(FR_state_wake, bins=bins_FR, color='green', label='Wake', alpha=0.5)
ax1.set_xlabel('FR (Hz)')
ax1.set_title('FR SLEEP-WAKE')
ax1 = plt.subplot2grid((2, 2), (0, 1), colspan=1)
ax1.hist(latency_sleep, bins=bins_latency, color='red', label='Sleep', alpha=1)
ax1.hist(latency_wake, bins=bins_latency, color='green', label='Wake', alpha=0.5)
ax1.set_xlabel('Latency (ms)')
ax1.set_title('Latency SLEEP-WAKE')
ax1 = plt.subplot2grid((2, 2), (1, 0), colspan=2)
ax1.scatter(latency_sleep, FR_state_sleep, label='Sleep', alpha=1, color='red')
ax1.scatter(latency_wake, FR_state_wake, label='Wake', alpha=0.5, color='green')
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('FR (Hz)')
ax1.set_title('Latency vs FR SLEEP-WAKE')

## Correlation analysis - extract PSTHs from metric dataframe according to significant metric

In [ ]:
sound_pick = [400]
pval_check = [-1, 0.01]
channels_pick = [0, 384]
stateOI = 'Onset'

df_Sleep = filter_dataframe(df, sound_pick, 'Sleep', channels_pick)
df_Wake = filter_dataframe(df, sound_pick, 'Wake', channels_pick)
df_Sleep, df_Wake = get_significant_df(df_Sleep, df_Wake, pval_check, stateOI)
df_Wake = df_Wake.sort_values('Channel', ascending=False, inplace=False).reset_index(drop=True)
df_Sleep = df_Sleep.sort_values('Channel', ascending=False, inplace=False).reset_index(drop=True)
# display(df_Sleep, df_Wake)

Cmat_channels = df_Sleep['Channel'].to_numpy()

In [ ]:
def plot_cmat(ax1, correlation_mat, labels):
    ticks_locs = np.arange(len(correlation_mat))
    
    im = ax1.imshow(gaussian_filter(correlation_mat,3), aspect='auto', origin='upper', cmap=plt.get_cmap('jet'), interpolation='nearest')
    
    cbar = plt.colorbar(im, ax=ax1, shrink=0.6)
    ax1.set_xticks(ticks_locs[::8], labels[::8])
    ax1.set_yticks(ticks_locs[::8], labels[::8])
    ax1.set_xlabel(f'Channels')
    ax1.set_ylabel(f'Channels')
    ax1.axvline(x=ticks_locs[np.argmin(np.abs(labels-120))])
    ax1.axhline(y=ticks_locs[np.argmin(np.abs(labels-120))])
    ax1.axvline(x=ticks_locs[np.argmin(np.abs(labels-92))])
    ax1.axhline(y=ticks_locs[np.argmin(np.abs(labels-92))])
    return im

### Signal Correlations - Correlation of the PSTH (averaged) response between units

In [ ]:
PSTHs_Sleep = np.stack(df_Sleep['PSTH'].to_numpy(), axis=0).mean(axis=1)
PSTHs_Wake = np.stack(df_Wake['PSTH'].to_numpy(), axis=0).mean(axis=1)

Cmat_Sleep_Signal = np.corrcoef(PSTHs_Sleep)
Cmat_Wake_Signal = np.corrcoef(PSTHs_Wake)

np.fill_diagonal(Cmat_Sleep_Signal, 0)
np.fill_diagonal(Cmat_Wake_Signal, 0)

In [ ]:
%matplotlib inline
plt.figure(figsize=(12, 9))
ax1 = plt.subplot2grid((2, 4), (0, 0), colspan=2)
im = plot_cmat(ax1, Cmat_Sleep_Signal, Cmat_channels)
ax1.set_title('Sleep')
# im.set_clim(np.min(Cmat_Sleep_Signal), np.max(Cmat_Sleep_Signal))
ax1 = plt.subplot2grid((2, 4), (0, 2), colspan=2)
im = plot_cmat(ax1, Cmat_Wake_Signal, Cmat_channels)
ax1.set_title('Wake')
# im.set_clim(np.min(Cmat_Sleep_Signal), np.max(Cmat_Sleep_Signal))
ax1 = plt.subplot2grid((2, 4), (1, 1), colspan=2)
im = plot_cmat(ax1, Cmat_Sleep_Signal - Cmat_Wake_Signal, Cmat_channels)
ax1.set_title('Sleep - Wake')
# im.set_clim(-0.4, 0.4)

plt.suptitle(f'{sound_pick} Signal')

### Noise Correlations - Correlation of the PSTH (per trial) response between units

In [ ]:
PSTHs_Sleep = np.stack(df_Sleep['PSTH'].to_numpy(), axis=0)
PSTHs_Wake = np.stack(df_Wake['PSTH'].to_numpy(), axis=0)

Cmat_Sleep_Noise = np.zeros((Cmat_channels.shape[0], Cmat_channels.shape[0]))
Cmat_Wake_Noise = np.zeros((Cmat_channels.shape[0], Cmat_channels.shape[0]))

for n in tqdm.tqdm(range(PSTHs_Sleep.shape[0])):
    neuron1_sleep = np.squeeze(PSTHs_Sleep[n,:,:])
    neuron1_wake = np.squeeze(PSTHs_Wake[n,:,:])
    for m in range(n+1, PSTHs_Sleep.shape[0]):
        neuron2_sleep = np.squeeze(PSTHs_Sleep[m,:,:])
        neuron2_wake = np.squeeze(PSTHs_Wake[m,:,:])
        
        ## Calculate the correlation for the pair of neurons trial wise
        sleep_corr = []
        for trial in range(neuron1_sleep.shape[0]):
            x = neuron1_sleep[trial,:]
            y = neuron2_sleep[trial,:]
            if (np.std(x) == 0) or (np.std(y) == 0):
                sleep_corr.append(np.nan)
            else:
                sleep_corr.append(np.corrcoef(x, y)[0, 1])
        
        wake_corr = []
        for trial in range(neuron1_wake.shape[0]):
            x = neuron1_wake[trial,:]
            y = neuron2_wake[trial,:]
            if (np.std(x) == 0) or (np.std(y) == 0):
                wake_corr.append(np.nan)
            else:
                wake_corr.append(np.corrcoef(x, y)[0, 1])
        
        Cmat_Sleep_Noise[n,m] = np.nanmean(sleep_corr)
        Cmat_Wake_Noise[n,m] = np.nanmean(wake_corr)
            
Cmat_Sleep_Noise = Cmat_Sleep_Noise + Cmat_Sleep_Noise.T - np.diag(np.diag(Cmat_Sleep_Noise))
Cmat_Wake_Noise = Cmat_Wake_Noise + Cmat_Wake_Noise.T - np.diag(np.diag(Cmat_Wake_Noise))

In [ ]:
plt.figure(figsize=(12, 9))
ax1 = plt.subplot2grid((2, 4), (0, 0), colspan=2)
im = plot_cmat(ax1, Cmat_Sleep_Noise, Cmat_channels)
im.set_clim(np.min(Cmat_Sleep_Noise)/10, np.max(Cmat_Sleep_Noise)/10)
ax1.set_title('Sleep')
ax1 = plt.subplot2grid((2, 4), (0, 2), colspan=2)
im = plot_cmat(ax1, Cmat_Wake_Noise, Cmat_channels)
im.set_clim(np.min(Cmat_Sleep_Noise), np.max(Cmat_Sleep_Noise))
ax1.set_title('Wake')
ax1 = plt.subplot2grid((2, 4), (1, 1), colspan=2)
im = plot_cmat(ax1, Cmat_Sleep_Noise - Cmat_Wake_Noise, Cmat_channels)
# im.set_clim(-0.04, 0.04)
ax1.set_title('Sleep - Wake')
plt.suptitle(f'{sound_pick} Noise')

### Convert Noise and Signal correlations to phase space

In [ ]:
Cmat_Sleep_Signal_phase = np.rad2deg(np.arctan(Cmat_Sleep_Signal / np.max(Cmat_Sleep_Signal)))
Cmat_Sleep_Noise_phase = np.rad2deg(np.arctan(Cmat_Sleep_Noise / np.max(Cmat_Sleep_Noise)))

diff_Phase_Sleep = Cmat_Sleep_Signal_phase - Cmat_Sleep_Noise_phase

plt.figure(figsize=(12, 9))
ax1 = plt.subplot2grid((2, 4), (0, 0), colspan=2)
im = plot_cmat(ax1, Cmat_Sleep_Signal_phase, Cmat_channels)
# im.set_clim(-60, 60)
ax1.set_title('Sleep Signal')
ax1 = plt.subplot2grid((2, 4), (0, 2), colspan=2)
im = plot_cmat(ax1, Cmat_Sleep_Noise_phase, Cmat_channels)
# im.set_clim(-30, 30)
ax1.set_title('Sleep Noise')
ax1 = plt.subplot2grid((2, 4), (1, 1), colspan=2)
im = plot_cmat(ax1, diff_Phase_Sleep, Cmat_channels)
# im.set_clim(-30, 30)
ax1.set_title('Signal - Noise')
plt.suptitle(f'{sound_pick} Noise')

In [ ]:
Cmat_Wake_Signal_phase = np.rad2deg(np.arctan(Cmat_Wake_Signal / np.max(Cmat_Wake_Signal)))
Cmat_Wake_Noise_phase = np.rad2deg(np.arctan(Cmat_Wake_Noise / np.max(Cmat_Wake_Noise)))

diff_Phase_Wake = Cmat_Wake_Signal_phase - Cmat_Wake_Noise_phase

plt.figure(figsize=(12, 9))
ax1 = plt.subplot2grid((2, 4), (0, 0), colspan=2)
im = plot_cmat(ax1, Cmat_Wake_Signal_phase, Cmat_channels)
# im.set_clim(-60, 60)
ax1.set_title('Wake Signal')
ax1 = plt.subplot2grid((2, 4), (0, 2), colspan=2)
im = plot_cmat(ax1, Cmat_Wake_Noise_phase, Cmat_channels)
# im.set_clim(-30, 30)
ax1.set_title('Wake Noise')
ax1 = plt.subplot2grid((2, 4), (1, 1), colspan=2)
im = plot_cmat(ax1, diff_Phase_Wake, Cmat_channels)
# im.set_clim(-30, 30)
ax1.set_title('Signal - Noise')
plt.suptitle(f'{sound_pick} Noise')

### PNAS Correlations - Spontaneous activity cross correlations

In [ ]:
PSTHs_Sleep = np.stack(df_Sleep['Spikes_raw'].to_numpy(), axis=0)
PSTHs_Wake = np.stack(df_Wake['Spikes_raw'].to_numpy(), axis=0)
PSTHs_Sleep_spont = PSTHs_Sleep[:,:,t_vec<0]
PSTHs_Wake_spont = PSTHs_Wake[:,:,t_vec<0]
t_vec_spont = t_vec[t_vec<0] + BUFFER_BEFORE_SEC

def cross_correlate_rasters(raster_a, raster_b, t_vec, t_vec_cross):
    bins = t_vec_cross
    correlogram = np.zeros(len(bins) - 1)
    
    th_val = np.max(raster_a)
    for trial in range(raster_a.shape[0]):
        spike_times1 = t_vec[np.where(raster_a[trial] == th_val)[0]]
        spike_times2 = t_vec[np.where(raster_b[trial] == th_val)[0]]
        
        for t1 in spike_times1:
            diffs = spike_times2 - t1
            valid_diffs = diffs[(diffs >= np.min(bins)) & (diffs <= np.max(bins))]
            correlogram += np.histogram(valid_diffs, bins=bins, density=False)[0]
    
    bin_centers = (bins[:-1] + bins[1:]) / 2
    correlogram = correlogram / raster_a.shape[0]
    
    return bin_centers, correlogram

def extract_delay_strength(bin_centers, correlogram):
    strength = np.max(correlogram) / np.diff(bin_centers).mean()
    delay = bin_centers[np.argmax(correlogram)] * 1000
    
    # if 1.5 < np.abs(delay) < 6:
    return delay, strength
    # else:
    #     return 0, 0

raster_A = np.squeeze(PSTHs_Sleep_spont[1, :, :])
raster_B = raster_A
bin_width = 0.001
t_vec_cross = np.arange(-0.05, 0.05 + bin_width, bin_width)


delay_map_sleep = np.zeros((PSTHs_Sleep_spont.shape[0], PSTHs_Sleep_spont.shape[0]))
strength_map_sleep = np.zeros_like(delay_map_sleep)

for ii in tqdm.tqdm(range(delay_map_sleep.shape[0])):
    raster_A = np.squeeze(PSTHs_Sleep_spont[ii, :, :])
    if np.sum(raster_A) == 0:
        continue
    for jj in range(ii+1, delay_map_sleep.shape[0]):
        raster_B = np.squeeze(PSTHs_Sleep_spont[jj, :, :])
        bin_centers, correlogram = cross_correlate_rasters(raster_A, raster_B, t_vec_spont, t_vec_cross)
        delay_map_sleep[ii,jj], strength_map_sleep[ii,jj] = extract_delay_strength(bin_centers, correlogram)

delay_map_sleep = delay_map_sleep + delay_map_sleep.T - np.diag(np.diag(delay_map_sleep))
strength_map_sleep = strength_map_sleep + strength_map_sleep.T - np.diag(np.diag(strength_map_sleep))

delay_map_wake = np.zeros((PSTHs_Wake_spont.shape[0], PSTHs_Wake_spont.shape[0]))
strength_map_wake = np.zeros_like(delay_map_wake)

for ii in tqdm.tqdm(range(delay_map_wake.shape[0])):
    raster_A = np.squeeze(PSTHs_Wake_spont[ii, :, :])
    if np.sum(raster_A) == 0:
        continue
    for jj in range(ii+1, delay_map_wake.shape[0]):
        raster_B = np.squeeze(PSTHs_Wake_spont[jj, :, :])
        bin_centers, correlogram = cross_correlate_rasters(raster_A, raster_B, t_vec_spont, t_vec_cross)
        delay_map_wake[ii,jj], strength_map_wake[ii,jj] = extract_delay_strength(bin_centers, correlogram)
        del bin_centers, correlogram

delay_map_wake = delay_map_wake + delay_map_wake.T - np.diag(np.diag(delay_map_wake))
strength_map_wake = strength_map_wake + strength_map_wake.T - np.diag(np.diag(strength_map_wake))

In [ ]:
%matplotlib inline

plt.figure(figsize=(12, 16))
ax1 = plt.subplot2grid((3, 4), (0, 0), colspan=2)
im = plot_cmat(ax1, delay_map_sleep, Cmat_channels)
# im.set_clim(-4, 4)
ax1.set_title('Delay map Sleep [ms]')
ax1 = plt.subplot2grid((3, 4), (0, 2), colspan=2)
im = plot_cmat(ax1, strength_map_sleep, Cmat_channels)
# im.set_clim(0, 0.2e6)
ax1.set_title('Strength Map Sleep [FR]')

ax1 = plt.subplot2grid((3, 4), (1, 0), colspan=2)
im = plot_cmat(ax1, delay_map_wake, Cmat_channels)
# im.set_clim(-4, 4)
ax1.set_title('Delay map Wake [ms]')
ax1 = plt.subplot2grid((3, 4), (1, 2), colspan=2)
im = plot_cmat(ax1, strength_map_wake, Cmat_channels)
# im.set_clim(0, 0.2e6)
ax1.set_title('Strength Map Wake [FR]')

ax1 = plt.subplot2grid((3, 4), (2, 0), colspan=2)
im = plot_cmat(ax1, delay_map_sleep - delay_map_wake, Cmat_channels)
# im.set_clim(-4, 4)
ax1.set_title('Delay map Diff [ms]')
ax1 = plt.subplot2grid((3, 4), (2, 2), colspan=2)
im = plot_cmat(ax1, strength_map_sleep - strength_map_wake, Cmat_channels)
# im.set_clim(0, 0.2e6)
ax1.set_title('Strength Map Diff [FR]')

plt.suptitle(f'{sound_pick} PNAS')